# Exercise 1: Initializing the Local Sentinel

In [1]:
!pip install -q -U kagglehub transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatibl

In [2]:
import torch
import kagglehub
from transformers import pipeline

MODEL_HANDLE = "google/gemma-4/Transformers/gemma-4-e2b-it/1"

# Download or attach the model from Kaggle.
model_path = kagglehub.model_download(MODEL_HANDLE)

print("Model path:", model_path)
print("GPU available:", torch.cuda.is_available())

# Initialize the local inference pipeline.
sentinel = pipeline(
    task="any-to-any",
    model=model_path,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

print("Gemma 4 E2B-IT initialized successfully.")

Model path: /kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1
GPU available: True


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Gemma 4 E2B-IT initialized successfully.


In [17]:
question = (
    "Why is this memory footprint reduction a critical factor "
    "when executing parallel physical simulations on the same hardware? "
    "Answer in 3 sentences."
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": question,
            }
        ],
    }
]

response = sentinel(
    messages,
    return_full_text=False,
    generate_kwargs={
        "max_new_tokens": 150,
        "do_sample": False,
    },
)

answer = response[0]["generated_text"]

print("Question:")
print(question)

print("\nGemma 4 Answer:")
print(answer)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Why is this memory footprint reduction a critical factor when executing parallel physical simulations on the same hardware? Answer in 3 sentences.

Gemma 4 Answer:
Reducing memory footprint is critical in parallel physical simulations because it allows more simulation data to reside in fast, on-chip memory, minimizing slow access to main memory. This directly improves computational efficiency by reducing latency and the need for frequent data transfers between processors. Consequently, a smaller footprint enables larger problem sizes or higher parallelism, leading to faster overall simulation times on the shared hardware.<turn|>


Question: Why is this memory footprint reduction a critical factor when executing parallel physical simulations on the same hardware?

Answer: Reducing memory footprint is critical in parallel physical simulations because it allows more simulation instances or larger datasets to fit within the available hardware memory. This prevents excessive swapping to slower storage, which would severely bottleneck the parallel execution speed. Ultimately, a smaller footprint ensures better resource utilization and faster, more efficient parallel computation.

## Exercise 2: Activating the Cognitive Core

In [4]:
import re
import torch

# Reuse the model already loaded in Exercise 1.
model = sentinel.model

# Depending on the pipeline version, the text processor may be exposed
# as either processor or tokenizer.
processor = getattr(sentinel, "processor", None)

if processor is None:
    processor = sentinel.tokenizer

print("Model and processor are ready.")

Model and processor are ready.


In [6]:
import re

THOUGHT_PATTERN = re.compile(
    r"<\|channel>thought\s*(.*?)<channel\|>",
    flags=re.DOTALL,
)


def extract_thought_channel(raw_response):
    """
    Extract the Gemma 4 thought channel.
    """
    match = THOUGHT_PATTERN.search(raw_response)

    if match is None:
        return ""

    return match.group(1).strip()


def remove_thought_channel(raw_response):
    """
    Remove the thought channel and remaining control tokens.
    """
    cleaned_response = THOUGHT_PATTERN.sub("", raw_response)

    control_tokens = [
        "<bos>",
        "<eos>",
        "<|turn>model",
        "<|turn>assistant",
        "<turn|>",
    ]

    for token in control_tokens:
        cleaned_response = cleaned_response.replace(token, "")

    return cleaned_response.strip()

In [7]:
def generate_thoughtful_response(user_prompt, max_new_tokens=1536):
    """
    Generate a response using Gemma 4 native thinking mode.

    Returns:
        raw_response: Complete decoded model output.
        thought: Extracted thought channel.
        answer: Final answer without the thought channel.
    """
    messages = [
        {
            "role": "user",
            "content": user_prompt,
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
        )

    generated_ids = output_ids[0][input_length:]

    raw_response = processor.decode(
        generated_ids,
        skip_special_tokens=False,
    )

    thought = extract_thought_channel(raw_response)
    answer = remove_thought_channel(raw_response)

    return {
        "raw_response": raw_response,
        "thought": thought,
        "answer": answer,
    }

In [8]:
question = (
    "Why is memory footprint reduction a critical factor when executing "
    "parallel physical simulations on the same hardware? "
    "Think briefly and provide the final answer in exactly three sentences."
)

result = generate_thoughtful_response(
    user_prompt=question,
    max_new_tokens=1536,
)

print("QUESTION:\n")
print(question)

print("\n" + "=" * 80)
print("EXTRACTED THOUGHT CHANNEL:\n")
print(result["thought"] or "No complete thought channel was detected.")

print("\n" + "=" * 80)
print("FINAL ANSWER:\n")
print(result["answer"] or "No final answer was generated.")

QUESTION:

Why is memory footprint reduction a critical factor when executing parallel physical simulations on the same hardware? Think briefly and provide the final answer in exactly three sentences.

EXTRACTED THOUGHT CHANNEL:

1.  **Analyze the Request:** The user is asking why memory footprint reduction is critical when running parallel physical simulations on the same hardware. The answer must be brief and exactly three sentences.

2.  **Identify Key Concepts:**
    *   Parallel physical simulations (e.g., fluid dynamics, climate models).
    *   Same hardware (shared resources).
    *   Memory footprint reduction (using less RAM/VRAM).
    *   Critical factor (why it matters).

3.  **Determine the Core Problems Caused by Large Footprints:**
    *   Resource contention (thrashing, swapping).
    *   Scaling limits (can't run enough tasks).
    *   Communication overhead (moving large datasets).
    *   Diminished performance (slowdown).

4.  **Draft the Three Sentences (Focusing o

In [11]:
def clean_conversation_history(history):
    """
    Remove Gemma thought blocks from every message before the history
    is reused in the next interaction.
    """
    cleaned_history = []

    for message in history:
        cleaned_message = message.copy()
        content = cleaned_message.get("content", "")

        if isinstance(content, str):
            cleaned_message["content"] = remove_thought_channel(content)

        elif isinstance(content, list):
            cleaned_content = []

            for item in content:
                cleaned_item = item.copy()

                if (
                    isinstance(cleaned_item, dict)
                    and cleaned_item.get("type") == "text"
                ):
                    cleaned_item["text"] = remove_thought_channel(
                        cleaned_item.get("text", "")
                    )

                cleaned_content.append(cleaned_item)

            cleaned_message["content"] = cleaned_content

        cleaned_history.append(cleaned_message)

    return cleaned_history

In [12]:
conversation_history = [
    {
        "role": "user",
        "content": question,
    },
    {
        "role": "assistant",
        "content": result["raw_response"],
    },
]

cleaned_history = clean_conversation_history(conversation_history)

print("ASSISTANT MESSAGE BEFORE CLEANING:\n")
print(conversation_history[1]["content"])

print("\n" + "=" * 80)
print("ASSISTANT MESSAGE AFTER CLEANING:\n")
print(cleaned_history[1]["content"])

ASSISTANT MESSAGE BEFORE CLEANING:

<|channel>thought
1.  **Analyze the Request:** The user is asking why memory footprint reduction is critical when running parallel physical simulations on the same hardware. The answer must be brief and exactly three sentences.

2.  **Identify Key Concepts:**
    *   Parallel physical simulations (e.g., fluid dynamics, climate models).
    *   Same hardware (shared resources).
    *   Memory footprint reduction (using less RAM/VRAM).
    *   Critical factor (why it matters).

3.  **Determine the Core Problems Caused by Large Footprints:**
    *   Resource contention (thrashing, swapping).
    *   Scaling limits (can't run enough tasks).
    *   Communication overhead (moving large datasets).
    *   Diminished performance (slowdown).

4.  **Draft the Three Sentences (Focusing on Impact):**
    *   *Sentence 1 (The Constraint):* Address the shared resource challenge.
    *   *Sentence 2 (The Mechanism):* Explain how high memory impacts performance (co

# Exercise 3: The Discrete Queue Simulation

In [14]:
import json
import numpy as np


class DiscreteFoundry:
    def __init__(
        self,
        arrival_rate,
        service_rate,
        capacity,
        initial_buffer=0,
        seed=42,
    ):
        if arrival_rate < 0:
            raise ValueError("arrival_rate must be non-negative.")

        if service_rate < 0:
            raise ValueError("service_rate must be non-negative.")

        if capacity <= 0:
            raise ValueError("capacity must be positive.")

        if not 0 <= initial_buffer <= capacity:
            raise ValueError(
                "initial_buffer must be between 0 and capacity."
            )

        self.arrival_rate = float(arrival_rate)
        self.service_rate = float(service_rate)
        self.capacity = int(capacity)

        self.buffer = int(initial_buffer)
        self.time_step = 0
        self.total_arrivals = 0
        self.total_processed = 0
        self.total_overflow = 0

        self.rng = np.random.default_rng(seed)

    def step(self):
        arrivals = int(self.rng.poisson(self.arrival_rate))
        service_attempts = int(self.rng.poisson(self.service_rate))

        available_space = self.capacity - self.buffer
        accepted_arrivals = min(arrivals, available_space)
        overflow = arrivals - accepted_arrivals

        self.buffer += accepted_arrivals

        processed = min(service_attempts, self.buffer)
        self.buffer -= processed

        self.time_step += 1
        self.total_arrivals += arrivals
        self.total_processed += processed
        self.total_overflow += overflow

        metrics = {
            "time_step": self.time_step,
            "arrivals": arrivals,
            "accepted_arrivals": accepted_arrivals,
            "processed": processed,
            "overflow": overflow,
            "buffer_level": self.buffer,
            "capacity": self.capacity,
            "total_arrivals": self.total_arrivals,
            "total_processed": self.total_processed,
            "total_overflow": self.total_overflow,
            "unstable": self.arrival_rate > self.service_rate,
        }

        return json.dumps(metrics)

In [15]:
stable_foundry = DiscreteFoundry(
    arrival_rate=3.0,
    service_rate=5.0,
    capacity=20,
    seed=42,
)

for _ in range(10):
    print(stable_foundry.step())

{"time_step": 1, "arrivals": 4, "accepted_arrivals": 4, "processed": 4, "overflow": 0, "buffer_level": 0, "capacity": 20, "total_arrivals": 4, "total_processed": 4, "total_overflow": 0, "unstable": false}
{"time_step": 2, "arrivals": 3, "accepted_arrivals": 3, "processed": 3, "overflow": 0, "buffer_level": 0, "capacity": 20, "total_arrivals": 7, "total_processed": 7, "total_overflow": 0, "unstable": false}
{"time_step": 3, "arrivals": 4, "accepted_arrivals": 4, "processed": 3, "overflow": 0, "buffer_level": 1, "capacity": 20, "total_arrivals": 11, "total_processed": 10, "total_overflow": 0, "unstable": false}
{"time_step": 4, "arrivals": 3, "accepted_arrivals": 3, "processed": 4, "overflow": 0, "buffer_level": 0, "capacity": 20, "total_arrivals": 14, "total_processed": 14, "total_overflow": 0, "unstable": false}
{"time_step": 5, "arrivals": 1, "accepted_arrivals": 1, "processed": 1, "overflow": 0, "buffer_level": 0, "capacity": 20, "total_arrivals": 15, "total_processed": 15, "total_ov

In [16]:
unstable_foundry = DiscreteFoundry(
    arrival_rate=8.0,
    service_rate=3.0,
    capacity=15,
    seed=42,
)

for _ in range(15):
    print(unstable_foundry.step())

{"time_step": 1, "arrivals": 10, "accepted_arrivals": 10, "processed": 4, "overflow": 0, "buffer_level": 6, "capacity": 15, "total_arrivals": 10, "total_processed": 4, "total_overflow": 0, "unstable": true}
{"time_step": 2, "arrivals": 10, "accepted_arrivals": 9, "processed": 0, "overflow": 1, "buffer_level": 15, "capacity": 15, "total_arrivals": 20, "total_processed": 4, "total_overflow": 1, "unstable": true}
{"time_step": 3, "arrivals": 8, "accepted_arrivals": 0, "processed": 3, "overflow": 8, "buffer_level": 12, "capacity": 15, "total_arrivals": 28, "total_processed": 7, "total_overflow": 9, "unstable": true}
{"time_step": 4, "arrivals": 9, "accepted_arrivals": 3, "processed": 0, "overflow": 6, "buffer_level": 15, "capacity": 15, "total_arrivals": 37, "total_processed": 7, "total_overflow": 15, "unstable": true}
{"time_step": 5, "arrivals": 10, "accepted_arrivals": 0, "processed": 4, "overflow": 10, "buffer_level": 11, "capacity": 15, "total_arrivals": 47, "total_processed": 11, "to

# Exercise 4: Closed-Loop Control with a Cognitive Entity

In [24]:
import re


CONTROLLER_INSTRUCTIONS = """
You are an autonomous industrial controller for a finite-capacity
manufacturing buffer.

Evaluate the supplied telemetry and select the service rate for the
next simulation step.

Control rules:
- The permitted service-rate interval is [0.5, 3.0].
- Increase the service rate when the buffer is growing or overflow occurs.
- Maintain or decrease the service rate when the buffer is low.
- Avoid unnecessary large changes.
- Perform your reasoning internally.
- Your final response must contain exactly one line:

NEW_SERVICE_RATE: X.X
""".strip()


SERVICE_RATE_PATTERN = re.compile(
    r"NEW[_\s-]*SERVICE[_\s-]*RATE\s*[:=]\s*"
    r"([0-9]+(?:\.[0-9]+)?)",
    flags=re.IGNORECASE,
)


def parse_service_rate(response_text):
    """
    Extract the numerical service rate from a model response.
    Only values in the required interval [0.5, 3.0] are accepted.
    """
    match = SERVICE_RATE_PATTERN.search(response_text)

    if match is None:
        return None

    new_rate = float(match.group(1))

    if not 0.5 <= new_rate <= 3.0:
        return None

    return new_rate

In [25]:
import torch


def generate_controller_decision(
    telemetry,
    current_service_rate,
    max_new_tokens=1024,
):
    """
    Ask Gemma 4 to evaluate the current telemetry and choose
    the service rate for the next simulation step.
    """
    controller_prompt = f"""
{CONTROLLER_INSTRUCTIONS}

CURRENT TELEMETRY
Arrival rate: {telemetry["arrival_rate"]}
Current service rate: {current_service_rate}
Buffer level: {telemetry["buffer_level"]}
Buffer capacity: {telemetry["capacity"]}
Arrivals this step: {telemetry["arrivals"]}
Processed this step: {telemetry["processed"]}
Overflow this step: {telemetry["overflow"]}
Total overflow: {telemetry["total_overflow"]}

Choose the service rate for the next step.

Return exactly:
NEW_SERVICE_RATE: X.X
""".strip()

    messages = [
        {
            "role": "user",
            "content": controller_prompt,
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = inputs.to(device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_ids = output_ids[0][input_length:]

    raw_response = processor.decode(
        generated_ids,
        skip_special_tokens=False,
    )

    # Remove the internal thought block before parsing the final answer.
    final_answer = remove_thought_channel(raw_response)

    # First try the cleaned final answer.
    new_service_rate = parse_service_rate(final_answer)

    # If necessary, also search the complete decoded response.
    if new_service_rate is None:
        new_service_rate = parse_service_rate(raw_response)

    parse_success = new_service_rate is not None

    # Safe fallback if the model does not respect the output format.
    if not parse_success:
        new_service_rate = current_service_rate

    # Final numerical safety check.
    new_service_rate = max(0.5, min(3.0, float(new_service_rate)))

    return {
        "raw_response": raw_response,
        "final_answer": final_answer,
        "new_service_rate": new_service_rate,
        "parse_success": parse_success,
    }

In [26]:
import json
import pandas as pd


controlled_foundry = DiscreteFoundry(
    arrival_rate=2.5,
    service_rate=1.0,
    capacity=10,
    seed=42,
)

control_log = []

for step_number in range(1, 9):
    old_service_rate = controlled_foundry.service_rate

    # Run one physical simulation step.
    telemetry_json = controlled_foundry.step()
    telemetry = json.loads(telemetry_json)

    # Add the rates required by the cognitive controller.
    telemetry["arrival_rate"] = controlled_foundry.arrival_rate
    telemetry["service_rate"] = old_service_rate

    # Ask Gemma 4 for the next control value.
    decision = generate_controller_decision(
        telemetry=telemetry,
        current_service_rate=old_service_rate,
        max_new_tokens=1024,
    )

    # Feed the decision back into the simulation.
    controlled_foundry.service_rate = decision["new_service_rate"]

    control_log.append(
        {
            "time_step": step_number,
            "arrivals": telemetry["arrivals"],
            "processed": telemetry["processed"],
            "buffer_level": telemetry["buffer_level"],
            "overflow": telemetry["overflow"],
            "old_service_rate": old_service_rate,
            "new_service_rate": decision["new_service_rate"],
            "parse_success": decision["parse_success"],
        }
    )

    print(f"TIME STEP {step_number}")
    print(f"Telemetry: {telemetry_json}")
    print(f"Controller output: {decision['final_answer']}")
    print(f"Applied service rate: {decision['new_service_rate']}")
    print(f"Parse successful: {decision['parse_success']}")
    print("-" * 80)

control_results = pd.DataFrame(control_log)
control_results

TIME STEP 1
Telemetry: {"time_step": 1, "arrivals": 4, "accepted_arrivals": 4, "processed": 3, "overflow": 0, "buffer_level": 1, "capacity": 10, "total_arrivals": 4, "total_processed": 3, "total_overflow": 0, "unstable": true}
Controller output: NEW_SERVICE_RATE: 2.5
Applied service rate: 2.5
Parse successful: True
--------------------------------------------------------------------------------
TIME STEP 2
Telemetry: {"time_step": 2, "arrivals": 4, "accepted_arrivals": 4, "processed": 2, "overflow": 0, "buffer_level": 3, "capacity": 10, "total_arrivals": 8, "total_processed": 5, "total_overflow": 0, "unstable": false}
Controller output: NEW_SERVICE_RATE: 3.0
Applied service rate: 3.0
Parse successful: True
--------------------------------------------------------------------------------
TIME STEP 3
Telemetry: {"time_step": 3, "arrivals": 0, "accepted_arrivals": 0, "processed": 3, "overflow": 0, "buffer_level": 0, "capacity": 10, "total_arrivals": 8, "total_processed": 8, "total_overflow

,time_step,arrivals,processed,buffer_level,overflow,old_service_rate,new_service_rate,parse_success
0,1,4,3,1,0,1.0,2.5,True
1,2,4,2,3,0,2.5,3.0,True
2,3,0,3,0,0,3.0,3.0,True
3,4,1,1,0,0,3.0,3.0,True
4,5,2,2,0,0,3.0,3.0,True
5,6,4,4,0,0,3.0,3.0,True
6,7,1,1,0,0,3.0,3.0,True
7,8,1,1,0,0,3.0,3.0,True


In [27]:
assert len(control_results) == 8

assert control_results["new_service_rate"].between(
    0.5,
    3.0,
).all()

print("Closed-loop simulation completed over eight time steps.")
print("All applied service rates remained within [0.5, 3.0].")

successful_parses = int(control_results["parse_success"].sum())

print(
    f"Successfully parsed model responses: "
    f"{successful_parses}/8"
)

if successful_parses == 8:
    print("All eight model decisions were parsed successfully.")
else:
    print(
        "The controller safely retained the previous service rate "
        "when a response could not be parsed."
    )

Closed-loop simulation completed over eight time steps.
All applied service rates remained within [0.5, 3.0].
Successfully parsed model responses: 8/8
All eight model decisions were parsed successfully.
